# 05 · Preprocessing for Clustering
## NHANES 2017–March 2020 Women's CKM Phenotyping Project

---

**Author:** Alexandra Velez, MD  
**Input:** `data/processed/ckm_features.csv` — 1,603 women aged 20–44  
**Output:** `data/processed/clustering_ready.csv` — scaled feature matrix 
ready for notebook 06  
**Last updated:** 2026

---

### Purpose of this notebook

This notebook implements the preprocessing decisions documented in notebook 
04 and prepares the final feature matrix for k-means clustering in notebook 
06. No new analytical decisions are made here — every step was motivated 
and justified in notebook 04.

The preprocessing pipeline follows this sequence:

1. **Drop collinear features** — Waist (r=0.95 with BMI) and Mean_DBP 
   (r=0.79 with Mean_SBP) removed, reducing the clustering feature set 
   from 12 to 10
2. **Resolve structural missingness** — never-pregnant women receive 
   Parity=0, Pregnancy_Loss=0, APO_Score=0 by clinical definition
3. **Impute residual missingness** — median imputation for Parity and 
   Pregnancy_Loss item non-response; KNN imputation (k=5) for Age_Menarche
4. **Exclude APO_Score unknowns** — 89 ever-pregnant women with unknown 
   APO status excluded from primary analysis, reducing sample to n=1,514
5. **Scale features** — RobustScaler applied to all 10 clustering features

### Decisions carried forward from notebook 04

| Decision | Rationale | Section |
|---|---|---|
| Drop Waist | Collinear with BMI, r=0.95 | §5 |
| Drop Mean_DBP | Collinear with Mean_SBP, r=0.79 | §5 |
| Zero-impute never-pregnant | Clinical definition, not imputation | §6 |
| Median impute Parity, Pregnancy_Loss | Small residual item non-response | §6 |
| KNN impute Age_Menarche | Known biological correlates in feature matrix | §6 |
| Exclude APO_Score NaN | Clinical events cannot be statistically estimated | §6 |
| Retain medicated women | Exclusion biases against highest-risk group | §7 |
| RobustScaler | Skewed distributions with meaningful clinical outliers | §8 |

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer
import warnings

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_PATH  = Path('../data/processed/ckm_features.csv')
OUTPUT_PATH = Path('../data/processed/clustering_ready.csv')
FIG_DIR     = Path('../figures')
FIG_DIR.mkdir(exist_ok=True)

# ── Clustering features — final 10 after collinearity decisions ────────────
# Waist and Mean_DBP dropped per notebook 04 Section 5
CLUSTERING_FEATURES = [
    'Age_Menarche',
    'Parity',
    'Pregnancy_Loss',
    'APO_Score',
    'HbA1c',
    'BMI',
    'Mean_SBP',
    'eGFR',
    'HDL',
    'Glucose',
]

# ── Plot style ─────────────────────────────────────────────────────────────
# Okabe-Ito colorblind-safe palette — consistent with prior notebooks
COLORS = {
    'no_apo':     '#0072B2',
    'macs_only':  '#56B4E9',
    'gdm_only':   '#E69F00',
    'gdm_macs':   '#D55E00',
    'never_preg': '#999999',
    'highlight':  '#CC79A7',
}

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')
print(f'  Input:   {INPUT_PATH}')
print(f'  Output:  {OUTPUT_PATH}')
print(f'  Figures: {FIG_DIR}')
print(f'\nClustering features ({len(CLUSTERING_FEATURES)}):')
for f in CLUSTERING_FEATURES:
    print(f'  {f}')

Setup complete.
  Input:   ../data/processed/ckm_features.csv
  Output:  ../data/processed/clustering_ready.csv
  Figures: ../figures

Clustering features (10):
  Age_Menarche
  Parity
  Pregnancy_Loss
  APO_Score
  HbA1c
  BMI
  Mean_SBP
  eGFR
  HDL
  Glucose


---
## Section 2 · Load & Validate

In [4]:
# ── Load CKM feature matrix ────────────────────────────────────────────────
df = pd.read_csv(INPUT_PATH, index_col='SEQN')

assert df.shape == (1603, 59), \
    f'Unexpected shape {df.shape} — expected (1603, 59)'
assert df.index.name == 'SEQN', \
    'SEQN is not the index'
assert all(f in df.columns for f in CLUSTERING_FEATURES), \
    'Some clustering features missing from input file'

print(f'✓ CKM feature matrix loaded: {df.shape}')
print(f'  Age range: {df["RIDAGEYR"].min():.0f}–{df["RIDAGEYR"].max():.0f} years')
print(f'  Ever-pregnant: {(df["Ever_Pregnant"] == 1).sum():,} women')
print(f'  Never-pregnant: {(df["Ever_Pregnant"] == 0).sum():,} women')

# ── Confirm clustering features present and check baseline missingness ──────
print(f'\nBaseline missingness — 10 clustering features:')
print(f'  {"Feature":<20} {"Valid":>6} {"NaN":>6} {"% Missing":>10}')
print(f'  {"─"*20} {"─"*6} {"─"*6} {"─"*10}')

for feat in CLUSTERING_FEATURES:
    n_valid   = df[feat].notna().sum()
    n_nan     = df[feat].isna().sum()
    pct_miss  = n_nan / len(df) * 100
    print(f'  {feat:<20} {n_valid:>6,} {n_nan:>6,} {pct_miss:>9.1f}%')

print(f'\n✓ All assertions passed — ready for preprocessing')

✓ CKM feature matrix loaded: (1603, 59)
  Age range: 20–44 years
  Ever-pregnant: 1,150 women
  Never-pregnant: 451 women

Baseline missingness — 10 clustering features:
  Feature               Valid    NaN  % Missing
  ──────────────────── ────── ────── ──────────
  Age_Menarche          1,525     78       4.9%
  Parity                1,135    468      29.2%
  Pregnancy_Loss        1,100    503      31.4%
  APO_Score             1,061    542      33.8%
  HbA1c                 1,522     81       5.1%
  BMI                   1,596      7       0.4%
  Mean_SBP              1,451    152       9.5%
  eGFR                  1,499    104       6.5%
  HDL                   1,503    100       6.2%
  Glucose               1,497    106       6.6%

✓ All assertions passed — ready for preprocessing


---
## Section 3 · Drop Collinear Features

Two features are dropped from the clustering set based on the collinearity 
analysis in notebook 04 Section 5. Both are retained in the full dataframe 
for descriptive characterization in notebook 07 — only their exclusion from 
the clustering feature set is enacted here.

- **Waist** — collinear with BMI (r = 0.95). BMI retained as the more 
  standard clinical measure of adiposity.
- **Mean_DBP** — collinear with Mean_SBP (r = 0.79). Mean_SBP retained 
  as the gold standard for long-term cardiovascular risk prediction.

In [6]:
# ── Drop collinear features from clustering set ────────────────────────────
# Waist and Mean_DBP remain in df for descriptive use in notebook 07
# They are excluded only from CLUSTERING_FEATURES, which was defined
# in Section 1 without them — no action needed on df itself

# ── Confirm neither feature is in the clustering set ──────────────────────
assert 'Waist' not in CLUSTERING_FEATURES, \
    'Waist should not be in clustering features'
assert 'Mean_DBP' not in CLUSTERING_FEATURES, \
    'Mean_DBP should not be in clustering features'

# ── Confirm both are still available in df for descriptive use ────────────
assert 'Waist' in df.columns, \
    'Waist missing from dataframe'
assert 'Mean_DBP' in df.columns, \
    'Mean_DBP missing from dataframe'

print('✓ Collinear features excluded from clustering set')
print(f'  Waist    — retained in df, excluded from clustering')
print(f'  Mean_DBP — retained in df, excluded from clustering')
print(f'\n✓ Clustering feature set confirmed: {len(CLUSTERING_FEATURES)} features')
print(f'  {CLUSTERING_FEATURES}')

✓ Collinear features excluded from clustering set
  Waist    — retained in df, excluded from clustering
  Mean_DBP — retained in df, excluded from clustering

✓ Clustering feature set confirmed: 10 features
  ['Age_Menarche', 'Parity', 'Pregnancy_Loss', 'APO_Score', 'HbA1c', 'BMI', 'Mean_SBP', 'eGFR', 'HDL', 'Glucose']


---
## Section 4 · Missingness Handling

Missingness in this dataset is predominantly structural — arising from 
NHANES skip logic rather than data collection failure. Each feature requires 
a different strategy based on the source of its missingness, as documented 
in notebook 04 Section 6.

The preprocessing steps in this section follow this order:

1. **Zero-impute never-pregnant women** — Parity, Pregnancy_Loss, and 
   APO_Score set to 0 for the 451 women who reported never having been 
   pregnant. This is a clinical definition, not statistical imputation.
2. **Median impute residual Parity and Pregnancy_Loss** — small item 
   non-response among ever-pregnant women (15 and 50 women respectively)
3. **Exclude APO_Score unknowns** — 89 ever-pregnant women with unknown 
   APO status excluded from primary analysis
4. **KNN impute Age_Menarche** — 78 women with random item non-response 
   imputed using k=5 nearest neighbors

In [8]:
# ── Step 1 — Zero-impute never-pregnant women ──────────────────────────────
# Clinical definition: a woman who has never been pregnant has zero
# deliveries, zero pregnancy losses, and zero adverse pregnancy outcomes
# This is not statistical imputation — it applies domain knowledge

df_pre = df.copy()   # preserve original df for reference

never_pregnant_mask = df_pre['Ever_Pregnant'] == 0

n_never = never_pregnant_mask.sum()

df_pre.loc[never_pregnant_mask, 'Parity']          = 0
df_pre.loc[never_pregnant_mask, 'Pregnancy_Loss']  = 0
df_pre.loc[never_pregnant_mask, 'APO_Score']       = 0

# ── Validate ───────────────────────────────────────────────────────────────
assert df_pre.loc[never_pregnant_mask, 'Parity'].isna().sum() == 0, \
    'NaN remains in Parity after zero-impute'
assert df_pre.loc[never_pregnant_mask, 'Pregnancy_Loss'].isna().sum() == 0, \
    'NaN remains in Pregnancy_Loss after zero-impute'
assert df_pre.loc[never_pregnant_mask, 'APO_Score'].isna().sum() == 0, \
    'NaN remains in APO_Score after zero-impute'

print(f'✓ Step 1 complete — zero-imputed {n_never:,} never-pregnant women')
print(f'  Parity          NaN remaining: '
      f'{df_pre["Parity"].isna().sum()}')
print(f'  Pregnancy_Loss  NaN remaining: '
      f'{df_pre["Pregnancy_Loss"].isna().sum()}')
print(f'  APO_Score       NaN remaining: '
      f'{df_pre["APO_Score"].isna().sum()}')

✓ Step 1 complete — zero-imputed 451 never-pregnant women
  Parity          NaN remaining: 17
  Pregnancy_Loss  NaN remaining: 52
  APO_Score       NaN remaining: 91


In [9]:
# ── Diagnose discrepancy ───────────────────────────────────────────────────
print('=== Ever_Pregnant value counts ===\n')
print(df['Ever_Pregnant'].value_counts(dropna=False))

print(f'\nEver_Pregnant NaN: {df["Ever_Pregnant"].isna().sum()}')

# Check if NaN Ever_Pregnant women account for the discrepancy
ep_nan_mask = df['Ever_Pregnant'].isna()
print(f'\nWomen with Ever_Pregnant NaN: {ep_nan_mask.sum()}')
print(f'  Parity NaN among them:         '
      f'{df.loc[ep_nan_mask, "Parity"].isna().sum()}')
print(f'  Pregnancy_Loss NaN among them: '
      f'{df.loc[ep_nan_mask, "Pregnancy_Loss"].isna().sum()}')
print(f'  APO_Score NaN among them:      '
      f'{df.loc[ep_nan_mask, "APO_Score"].isna().sum()}')

=== Ever_Pregnant value counts ===

Ever_Pregnant
1.0    1150
0.0     451
NaN       2
Name: count, dtype: int64

Ever_Pregnant NaN: 2

Women with Ever_Pregnant NaN: 2
  Parity NaN among them:         2
  Pregnancy_Loss NaN among them: 2
  APO_Score NaN among them:      2


### Step 1 · Zero-imputation for never-pregnant women

The 451 women who reported never having been pregnant received zero 
values for Parity, Pregnancy_Loss, and APO_Score. This is a clinical 
definition — a woman who has never been pregnant has zero deliveries, 
zero pregnancy losses, and zero adverse pregnancy outcomes by definition. 
No statistical estimation is involved.

After zero-imputation, a discrepancy was identified: residual NaN counts 
were 17, 52, and 91 rather than the 15, 50, and 89 documented in notebook 
04. Investigation revealed 2 women with unknown pregnancy status 
(Ever_Pregnant = NaN) — they did not answer the gate question and were 
therefore captured by neither the never-pregnant nor the ever-pregnant 
mask in the notebook 04 diagnostic.

These 2 women cannot be zero-imputed (pregnancy status unknown) and 
cannot be classified as ever-pregnant. They are excluded from the 
analysis — a negligible sample loss that reduces the working sample 
from 1,603 to 1,601. After their exclusion, residual NaN counts return 
to the expected values: Parity 15, Pregnancy_Loss 50, APO_Score 89.

In [10]:
# ── Handle Ever_Pregnant NaN — 2 women ────────────────────────────────────
# Pregnancy status unknown — cannot apply zero-impute or ever-pregnant logic
# Excluded from analysis; 2 women is a negligible sample loss

ep_nan_mask = df_pre['Ever_Pregnant'].isna()
df_pre      = df_pre[~ep_nan_mask].copy()

assert len(df_pre) == 1601, \
    f'Unexpected row count after exclusion: {len(df_pre)}'

print(f'✓ 2 women with unknown pregnancy status excluded')
print(f'  Remaining sample: {len(df_pre):,} women')
print(f'\n  Parity NaN remaining:         '
      f'{df_pre["Parity"].isna().sum()}')
print(f'  Pregnancy_Loss NaN remaining: '
      f'{df_pre["Pregnancy_Loss"].isna().sum()}')
print(f'  APO_Score NaN remaining:      '
      f'{df_pre["APO_Score"].isna().sum()}')

✓ 2 women with unknown pregnancy status excluded
  Remaining sample: 1,601 women

  Parity NaN remaining:         15
  Pregnancy_Loss NaN remaining: 50
  APO_Score NaN remaining:      89


### Step 2 · Median imputation — Parity and Pregnancy_Loss

Among ever-pregnant women, 15 had missing Parity and 50 had missing Pregnancy_Loss due to item non-response. Median imputation was applied within the ever-pregnant subgroup only — never-pregnant women already received zero values in Step 1 and were not affected. The median Parity among ever-pregnant women was 2.0 deliveries and the median Pregnancy_Loss was 0.0 — both clinically sensible values for this sample. Both features are now fully imputed with zero NaN remaining.

In [11]:
# ── Step 2 — Median impute Parity and Pregnancy_Loss ──────────────────────
# 15 and 50 ever-pregnant women respectively with item non-response
# Median imputation within ever-pregnant women only
# Never-pregnant women already have 0 from Step 1 — not affected

ever_pregnant_mask = df_pre['Ever_Pregnant'] == 1

for feat in ['Parity', 'Pregnancy_Loss']:
    median_val = df_pre.loc[ever_pregnant_mask, feat].median()
    n_imputed  = df_pre.loc[ever_pregnant_mask, feat].isna().sum()

    df_pre.loc[ever_pregnant_mask & df_pre[feat].isna(), feat] = median_val

    print(f'✓ {feat:<20} median = {median_val:.1f}  '
          f'({n_imputed} values imputed)')

# ── Validate ───────────────────────────────────────────────────────────────
assert df_pre['Parity'].isna().sum() == 0, \
    'NaN remains in Parity after median imputation'
assert df_pre['Pregnancy_Loss'].isna().sum() == 0, \
    'NaN remains in Pregnancy_Loss after median imputation'

print(f'\n✓ Step 2 complete — Parity and Pregnancy_Loss fully imputed')
print(f'  Parity NaN remaining:         0')
print(f'  Pregnancy_Loss NaN remaining: 0')

✓ Parity               median = 2.0  (15 values imputed)
✓ Pregnancy_Loss       median = 0.0  (50 values imputed)

✓ Step 2 complete — Parity and Pregnancy_Loss fully imputed
  Parity NaN remaining:         0
  Pregnancy_Loss NaN remaining: 0


### Step 3 · Exclude APO_Score unknowns

89 ever-pregnant women did not answer the gestational diabetes or 
macrosomia questions, leaving their APO_Score unknown. These women cannot 
be imputed — APO_Score encodes discrete clinical events that either 
occurred or did not. Assigning a statistical estimate to unknown pregnancy 
outcomes would be clinically indefensible and could misclassify women into 
APO-positive or APO-negative clusters without any evidentiary basis.

These 89 women are excluded from the primary clustering analysis and 
preserved in a separate dataframe (df_apo_unknown) for descriptive 
characterization in notebook 07. Combined with the 2 women excluded in 
Step 1 for unknown pregnancy status, the primary analysis sample is 
reduced from 1,603 to 1,512 women — a total exclusion of 91 women (5.7%).

In [13]:
# ── Step 3 — Exclude APO_Score unknowns ───────────────────────────────────
# 89 ever-pregnant women did not answer GDM or macrosomia questions
# APO_Score encodes discrete clinical events that cannot be statistically
# estimated — exclusion is the only clinically defensible approach
# These women are preserved in a separate dataframe for descriptive
# characterization in notebook 07

apo_unknown_mask = df_pre['APO_Score'].isna()
n_excluded       = apo_unknown_mask.sum()

# Preserve for descriptive characterization
df_apo_unknown = df_pre[apo_unknown_mask].copy()

# Primary analysis sample
df_pre = df_pre[~apo_unknown_mask].copy()

assert len(df_pre) == 1512, \
    f'Unexpected row count after APO exclusion: {len(df_pre)}'
assert df_pre['APO_Score'].isna().sum() == 0, \
    'APO_Score NaN remains after exclusion'

print(f'✓ Step 3 complete — APO_Score unknowns excluded')
print(f'  Excluded:                {n_excluded} women')
print(f'  Preserved in df_apo_unknown for notebook 07 characterization')
print(f'  Primary analysis sample: {len(df_pre):,} women')
print(f'  APO_Score NaN remaining: {df_pre["APO_Score"].isna().sum()}')

# ── APO distribution in primary sample ────────────────────────────────────
print(f'\nAPO distribution in primary sample (n={len(df_pre):,}):')
print(f'  {"Group":<25} {"N":>6} {"% of sample":>12}')
print(f'  {"─"*25} {"─"*6} {"─"*12}')

never_preg_mask = df_pre['Ever_Pregnant'] == 0
ever_preg_mask  = df_pre['Ever_Pregnant'] == 1

n_never = never_preg_mask.sum()
print(f'  {"Never pregnant":<25} {n_never:>6,} {n_never/len(df_pre)*100:>11.1f}%')

apo_map = {
    0.0: 'No APOs',
    1.0: 'Macrosomia only',
    2.0: 'GDM only',
    3.0: 'GDM + Macrosomia',
}

n_ever_total = 0
for score, label in apo_map.items():
    n = (ever_preg_mask & (df_pre['APO_Score'] == score)).sum()
    n_ever_total += n
    print(f'  {label:<25} {n:>6,} {n/len(df_pre)*100:>11.1f}%')

print(f'  {"─"*25} {"─"*6} {"─"*12}')
print(f'  {"Total":<25} {len(df_pre):>6,} {"100.0%":>12}')

✓ Step 3 complete — APO_Score unknowns excluded
  Excluded:                0 women
  Preserved in df_apo_unknown for notebook 07 characterization
  Primary analysis sample: 1,512 women
  APO_Score NaN remaining: 0

APO distribution in primary sample (n=1,512):
  Group                          N  % of sample
  ───────────────────────── ────── ────────────
  Never pregnant               451        29.8%
  No APOs                      813        53.8%
  Macrosomia only              114         7.5%
  GDM only                     104         6.9%
  GDM + Macrosomia              30         2.0%
  ───────────────────────── ────── ────────────
  Total                      1,512       100.0%


The final APO distribution in the primary sample is mutually exclusive 
and clinically coherent: 451 never-pregnant women (29.8%), 813 with no 
APOs (53.8%), and 248 with at least one APO (16.4%) — the primary 
exposure group for the clustering analysis.

### Step 4 · KNN imputation — Age_Menarche

78 women in the original sample had missing Age_Menarche due to random 
item non-response (1 additional was lost in the Step 1 exclusion, leaving 
77 to impute in the primary sample). KNN imputation with k=5 neighbors was 
applied across the full 10-feature clustering set — all features informed 
neighbor selection, producing biologically grounded estimates rather than 
a uniform population average.

In [18]:
# ── Step 4 — KNN imputation for Age_Menarche ──────────────────────────────
# 78 women with random item non-response on Age_Menarche
# KNN (k=5) leverages known biological associations between menarche age
# and BMI, race/ethnicity, and other features already in the matrix
# Imputation is performed on the full 10-feature clustering set

# Check Age_Menarche missingness in primary sample before imputation
n_missing_before = df_pre['Age_Menarche'].isna().sum()
print(f'Age_Menarche NaN before imputation: {n_missing_before}')

# ── KNN imputation ─────────────────────────────────────────────────────────
# KNNImputer operates on the full feature matrix — all 10 clustering
# features inform neighbor selection, producing biologically grounded
# estimates for missing Age_Menarche values

imputer = KNNImputer(n_neighbors=5)

# Extract clustering features for imputation
X_impute = df_pre[CLUSTERING_FEATURES].copy()

# Fit and transform
X_imputed = imputer.fit_transform(X_impute)

# Put back into dataframe
df_imputed = pd.DataFrame(
    X_imputed,
    index=df_pre.index,
    columns=CLUSTERING_FEATURES
)

# Update Age_Menarche in df_pre with imputed values
df_pre['Age_Menarche'] = df_imputed['Age_Menarche']

# ── Validate ───────────────────────────────────────────────────────────────
n_missing_after = df_pre['Age_Menarche'].isna().sum()

assert n_missing_after == 0, \
    f'NaN remains in Age_Menarche after KNN imputation'

print(f'Age_Menarche NaN after imputation:  {n_missing_after}')
print(f'\n✓ Step 4 complete — Age_Menarche KNN imputed')
print(f'  k = 5 neighbors')
print(f'  Values imputed: {n_missing_before}')
print(f'\nImputed value distribution:')
print(f'  Min:    {df_pre["Age_Menarche"].min():.1f} years')
print(f'  Median: {df_pre["Age_Menarche"].median():.1f} years')
print(f'  Max:    {df_pre["Age_Menarche"].max():.1f} years')
print(f'  Mean:   {df_pre["Age_Menarche"].mean():.2f} years')

Age_Menarche NaN before imputation: 0
Age_Menarche NaN after imputation:  0

✓ Step 4 complete — Age_Menarche KNN imputed
  k = 5 neighbors
  Values imputed: 0

Imputed value distribution:
  Min:    6.0 years
  Median: 12.0 years
  Max:    20.0 years
  Mean:   12.60 years


In [20]:
# ── Check implausible imputed values ──────────────────────────────────────
# Identify imputed women by comparing df_pre to original df on same index
original_age_men = df.loc[df_pre.index, 'Age_Menarche']
imputed_indices  = original_age_men[original_age_men.isna()].index
imputed_values   = df_pre.loc[imputed_indices, 'Age_Menarche']

print(f'Imputed Age_Menarche distribution (n={len(imputed_values)}):')
print(f'  Min:    {imputed_values.min():.1f}')
print(f'  Max:    {imputed_values.max():.1f}')
print(f'  Mean:   {imputed_values.mean():.2f}')
print(f'  Median: {imputed_values.median():.1f}')

print(f'\nValues below age 8 (implausible):')
implausible = imputed_values[imputed_values < 8]
print(f'  N = {len(implausible)}')
if len(implausible) > 0:
    print(f'  Values: {sorted(implausible.values)}')

Imputed Age_Menarche distribution (n=77):
  Min:    11.0
  Max:    14.2
  Mean:   12.60
  Median: 12.6

Values below age 8 (implausible):
  N = 0


In [21]:
# ── Check observed Age_Menarche outliers ───────────────────────────────────
observed_indices  = original_age_men[original_age_men.notna()].index
observed_values   = df_pre.loc[observed_indices, 'Age_Menarche']

print(f'Observed Age_Menarche distribution (n={len(observed_values):,}):')
print(f'  Min:    {observed_values.min():.1f}')
print(f'  Max:    {observed_values.max():.1f}')
print(f'  Mean:   {observed_values.mean():.2f}')
print(f'  Median: {observed_values.median():.1f}')

print(f'\nValues below age 8 (clinically implausible):')
implausible_obs = observed_values[observed_values < 8]
print(f'  N = {len(implausible_obs)}')
if len(implausible_obs) > 0:
    print(f'  Values: {sorted(implausible_obs.values)}')

Observed Age_Menarche distribution (n=1,435):
  Min:    6.0
  Max:    20.0
  Mean:   12.60
  Median: 12.0

Values below age 8 (clinically implausible):
  N = 1
  Values: [np.float64(6.0)]


In [22]:
# ── Clip implausible Age_Menarche value ────────────────────────────────────
# 1 woman with observed Age_Menarche = 6.0 — not physiologically plausible
# Precocious puberty is defined as menarche before age 8
# Value clipped to 8.0 — the clinical floor for menarche age
# This affects 1 woman (0.07% of the primary sample)

n_clipped = (df_pre['Age_Menarche'] < 8).sum()

df_pre.loc[df_pre['Age_Menarche'] < 8, 'Age_Menarche'] = 8.0

assert (df_pre['Age_Menarche'] < 8).sum() == 0, \
    'Values below 8 remain after clipping'

print(f'✓ Age_Menarche clipped to physiological floor of 8.0 years')
print(f'  Values clipped: {n_clipped} (0.07% of primary sample)')
print(f'\nFinal Age_Menarche distribution:')
print(f'  Min:    {df_pre["Age_Menarche"].min():.1f}')
print(f'  Max:    {df_pre["Age_Menarche"].max():.1f}')
print(f'  Mean:   {df_pre["Age_Menarche"].mean():.2f}')
print(f'  Median: {df_pre["Age_Menarche"].median():.1f}')

✓ Age_Menarche clipped to physiological floor of 8.0 years
  Values clipped: 1 (0.07% of primary sample)

Final Age_Menarche distribution:
  Min:    8.0
  Max:    20.0
  Mean:   12.60
  Median: 12.0


Imputed values ranged from 11.0 to 14.2 years with a median of 12.6 — 
entirely within the physiologically plausible range for menarche onset. 
The minimum of 6.0 years visible in the overall distribution output above 
belongs to an observed value, not an imputed one — no implausible values 
were introduced by the imputation.

**Data quality finding — one observed value clipped:**
Inspection of the observed Age_Menarche distribution identified one woman 
with a recorded menarche age of 6.0 years. This is not physiologically 
plausible — precocious puberty is clinically defined as menarche before 
age 8, and age 6 represents an extreme that almost certainly reflects 
survey response error. This value was clipped to 8.0 years, the accepted 
clinical floor for menarche age. This affects 1 woman (0.07% of the 
primary sample) and has no meaningful impact on the feature distribution 
— the mean and median are unchanged at 12.60 and 12.0 years respectively.

After Steps 1–4, all reproductive clustering features are fully imputed 
with zero NaN remaining. The primary analysis sample stands at 1,512 women.

---
## Section 5 · Feature Scaling

K-means clustering uses Euclidean distance to assign observations to 
clusters. Features on larger numeric scales contribute disproportionately 
to distance calculations regardless of their clinical importance — without 
scaling, eGFR (range ~60–145) and Glucose (range ~60–200) would dominate 
cluster assignments while APO_Score (range 0–3) would contribute almost 
nothing. Scaling is therefore not optional; it is a prerequisite for 
meaningful clustering.

### Why not z-score standardization?

Z-score standardization (StandardScaler) is the most commonly used scaling 
method and the default choice in most machine learning workflows. It 
transforms each feature to mean = 0 and standard deviation = 1 using the 
formula (x − mean) / std. For normally distributed features without 
extreme values, it performs well and produces easily interpretable results.

However, z-score standardization has a fundamental vulnerability: both the 
mean and the standard deviation are sensitive to outliers. In a clinical 
dataset like this one, extreme values are not errors — they are real patients 
with real disease. A woman with HbA1c of 13.0% has uncontrolled diabetes. 
A woman with BMI of 58 has severe obesity. A woman with Glucose of 380 mg/dL 
is in metabolic crisis. These values are clinically meaningful and must be 
retained in the analysis.

The problem is that when z-score scaling is applied, these extreme values 
inflate the standard deviation of their respective features. A larger 
standard deviation compresses the scaled values for the majority of women 
into a narrower band, reducing their contribution to distance calculations 
and giving disproportionate influence to the outliers. In effect, z-score 
scaling in a skewed clinical distribution does the opposite of what scaling 
is supposed to do — instead of equalizing feature contributions, it 
amplifies the influence of the most extreme cases.

This is not a theoretical concern. The biomarker distributions examined in 
notebook 04 confirmed right-skewed distributions with meaningful clinical 
outliers across HbA1c, BMI, Glucose, and eGFR. Z-score scaling is 
therefore not appropriate for this dataset.

### Robust scaling

RobustScaler transforms each feature using the median and interquartile 
range (IQR): (x − median) / IQR. Because the median and IQR are 
order-based statistics, they are inherently resistant to extreme values — 
an outlier at either tail does not affect the median or IQR of a large 
distribution. The scaling parameters therefore reflect the central 
tendency and spread of the majority of the sample, not the extremes.

This has two important advantages for this dataset:

First, the scaling parameters themselves are clinically meaningful. The 
median and IQR of HbA1c, BMI, and Mean_SBP in this sample are 
interpretable statistics that a clinician can evaluate directly — unlike 
the mean and standard deviation, which are distorted by the skewed 
distributions documented in notebook 04.

Second, the reproductive features engineered in notebook 02 have 
non-normal distributions by design. APO_Score is zero-inflated — 76.6% 
of ever-pregnant women scored 0. Parity and Pregnancy_Loss are similarly 
zero-heavy after zero-imputing never-pregnant women. Robust scaling 
handles these distributions more gracefully than z-score, preserving the 
meaningful variation in the non-zero values without being dominated by 
the zero mass.

**Decision: RobustScaler is applied to all 10 clustering features.**

Cluster stability under StandardScaler is examined as part of the 
sensitivity analysis in Section 7 to confirm that the choice of scaling 
method does not materially alter the cluster structure.

In [23]:
# ── Feature scaling — RobustScaler ────────────────────────────────────────
# Applied to all 10 clustering features in the primary analysis sample
# Median and IQR computed on the primary sample (n=1,512)
# Women with NaN on any CKM biomarker are handled after scaling —
# k-means will operate on complete cases only

# ── Extract clustering features ────────────────────────────────────────────
X = df_pre[CLUSTERING_FEATURES].copy()

print(f'Feature matrix before scaling:')
print(f'  Shape: {X.shape}')
print(f'  Complete cases: {X.dropna().shape[0]:,}')
print(f'  Cases with any NaN: {X.shape[0] - X.dropna().shape[0]:,}')

# ── Fit and transform ──────────────────────────────────────────────────────
scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(
    X_scaled,
    index=df_pre.index,
    columns=CLUSTERING_FEATURES
)

# ── Scaling parameters — document for reproducibility ─────────────────────
print(f'\nScaling parameters (median / IQR):')
print(f'  {"Feature":<20} {"Median":>10} {"IQR":>10}')
print(f'  {"─"*20} {"─"*10} {"─"*10}')

for i, feat in enumerate(CLUSTERING_FEATURES):
    median = scaler.center_[i]
    iqr    = scaler.scale_[i]
    print(f'  {feat:<20} {median:>10.3f} {iqr:>10.3f}')

# ── Validate scaled distribution ───────────────────────────────────────────
print(f'\nScaled feature ranges (should be centered near 0):')
print(f'  {"Feature":<20} {"Min":>8} {"Median":>8} {"Max":>8}')
print(f'  {"─"*20} {"─"*8} {"─"*8} {"─"*8}')

for feat in CLUSTERING_FEATURES:
    vals = df_scaled[feat].dropna()
    print(f'  {feat:<20} {vals.min():>8.2f} {vals.median():>8.2f} '
          f'{vals.max():>8.2f}')

print(f'\n✓ RobustScaler applied to all {len(CLUSTERING_FEATURES)} features')

Feature matrix before scaling:
  Shape: (1512, 10)
  Complete cases: 1,285
  Cases with any NaN: 227

Scaling parameters (median / IQR):
  Feature                  Median        IQR
  ──────────────────── ────────── ──────────
  Age_Menarche             12.000      1.000
  Parity                    2.000      3.000
  Pregnancy_Loss            0.000      1.000
  APO_Score                 0.000      1.000
  HbA1c                     5.300      0.400
  BMI                      28.800     11.400
  Mean_SBP                108.000     16.000
  eGFR                    113.573     20.180
  HDL                      54.000     19.000
  Glucose                  88.000     11.000

Scaled feature ranges (should be centered near 0):
  Feature                   Min   Median      Max
  ──────────────────── ──────── ──────── ────────
  Age_Menarche            -4.00     0.00     8.00
  Parity                  -0.67     0.00     1.00
  Pregnancy_Loss           0.00     0.00    10.00
  APO_Score          

### Scaling results

RobustScaler was applied to all 10 clustering features using the median 
and IQR computed on the primary sample (n=1,512). All scaled features are 
centered at 0 as expected — confirming the scaler was applied correctly.

**Scaling parameters are clinically interpretable:**
The median values reflect the central tendency of this reproductive-age 
sample. Median HbA1c of 5.3% and median glucose of 88 mg/dL are both 
within normal range — consistent with a predominantly healthy young female 
sample. Median BMI of 28.8 sits at the overweight threshold, and median 
SBP of 108 mmHg reflects the characteristically lower blood pressure of 
reproductive-age women.

**Asymmetric scaled ranges in reproductive features:**
Pregnancy_Loss and APO_Score both have a scaled minimum of 0.0. This is 
expected — the median of both features is 0, so all zero values (the 
majority of the sample after zero-imputing never-pregnant women) scale to 
exactly 0. The non-zero values extend rightward only, reflecting the 
zero-inflated nature of these features. This asymmetry is a property of 
the data, not a scaling artifact.

**Extreme right tails in metabolic features:**
HbA1c reaches a scaled maximum of 23.75 and Glucose of 26.27 — reflecting 
women with severely uncontrolled diabetes at the far right of the 
distribution. These are real clinical values retained intentionally. Under 
z-score scaling, these extreme values would have inflated the standard 
deviation and distorted the scaling for the majority of the sample — the 
primary reason RobustScaler was chosen.

**Effective clustering sample:**
Of the 1,512 women in the primary sample, 227 have missing values on at 
least one CKM biomarker due to incomplete examination module participation. 
K-means requires complete cases — the impact on effective sample size is 
quantified in the next cell.

In [24]:
# ── Complete cases quantification ──────────────────────────────────────────
# K-means requires complete cases — quantify impact of CKM biomarker
# missingness on effective clustering sample

print('=== Complete Cases Analysis ===\n')

# ── Overall complete cases ─────────────────────────────────────────────────
n_total    = len(df_scaled)
n_complete = df_scaled.dropna().shape[0]
n_missing  = n_total - n_complete

print(f'  Primary sample:          {n_total:,}')
print(f'  Complete cases:          {n_complete:,} ({n_complete/n_total*100:.1f}%)')
print(f'  Incomplete cases:        {n_missing:,} ({n_missing/n_total*100:.1f}%)')

# ── Missingness by feature ─────────────────────────────────────────────────
print(f'\n  Missingness by CKM biomarker:')
print(f'  {"Feature":<20} {"NaN":>6} {"% Missing":>10}')
print(f'  {"─"*20} {"─"*6} {"─"*10}')

ckm_biomarkers = ['HbA1c', 'BMI', 'Mean_SBP', 'eGFR', 'HDL', 'Glucose']
for feat in ckm_biomarkers:
    n_nan = df_scaled[feat].isna().sum()
    pct   = n_nan / n_total * 100
    print(f'  {feat:<20} {n_nan:>6,} {pct:>9.1f}%')

# ── Reproductive features should have zero NaN at this point ──────────────
print(f'\n  Reproductive features (should be 0 NaN after preprocessing):')
repro = ['Age_Menarche', 'Parity', 'Pregnancy_Loss', 'APO_Score']
for feat in repro:
    n_nan = df_scaled[feat].isna().sum()
    print(f'  {feat:<20} {n_nan:>6,} NaN')

# ── APO distribution among complete cases ─────────────────────────────────
df_complete = df_scaled.dropna().copy()
df_complete['Ever_Pregnant'] = df_pre.loc[df_complete.index, 'Ever_Pregnant']
df_complete['APO_Score_raw'] = df_pre.loc[df_complete.index, 'APO_Score']

print(f'\n  APO distribution among complete cases (n={n_complete:,}):')
print(f'  {"Group":<25} {"N":>6} {"% of complete":>14}')
print(f'  {"─"*25} {"─"*6} {"─"*14}')

never_preg_mask = df_complete['Ever_Pregnant'] == 0
n_never = never_preg_mask.sum()
print(f'  {"Never pregnant":<25} {n_never:>6,} '
      f'{n_never/n_complete*100:>13.1f}%')

apo_map = {
    0.0: 'No APOs',
    1.0: 'Macrosomia only',
    2.0: 'GDM only',
    3.0: 'GDM + Macrosomia',
}
ever_preg_mask = df_complete['Ever_Pregnant'] == 1
for score, label in apo_map.items():
    n = (ever_preg_mask & (df_complete['APO_Score_raw'] == score)).sum()
    print(f'  {label:<25} {n:>6,} {n/n_complete*100:>13.1f}%')

print(f'  {"─"*25} {"─"*6} {"─"*14}')
print(f'  {"Total":<25} {n_complete:>6,} {"100.0%":>14}')

# ── Assertion ──────────────────────────────────────────────────────────────
assert n_complete >= 1200, \
    f'Complete cases too low for reliable clustering: {n_complete}'

print(f'\n✓ Effective clustering sample: {n_complete:,} women')
print(f'  ({n_missing} excluded due to missing CKM biomarkers — '
      f'{n_missing/n_total*100:.1f}% of primary sample)')

=== Complete Cases Analysis ===

  Primary sample:          1,512
  Complete cases:          1,285 (85.0%)
  Incomplete cases:        227 (15.0%)

  Missingness by CKM biomarker:
  Feature                 NaN  % Missing
  ──────────────────── ────── ──────────
  HbA1c                    71       4.7%
  BMI                       7       0.5%
  Mean_SBP                143       9.5%
  eGFR                     91       6.0%
  HDL                      88       5.8%
  Glucose                  93       6.2%

  Reproductive features (should be 0 NaN after preprocessing):
  Age_Menarche              0 NaN
  Parity                    0 NaN
  Pregnancy_Loss            0 NaN
  APO_Score                 0 NaN

  APO distribution among complete cases (n=1,285):
  Group                          N  % of complete
  ───────────────────────── ────── ──────────────
  Never pregnant               391          30.4%
  No APOs                      680          52.9%
  Macrosomia only               97       

### Complete cases analysis

Of the 1,512 women in the primary sample, 1,285 (85.0%) have complete 
data across all 10 clustering features and form the effective clustering 
sample for notebook 06. The 227 incomplete cases (15.0%) are missing at 
least one CKM biomarker due to incomplete examination module participation 
— the most common source being Mean_SBP (143 women, 9.5%), followed by 
Glucose (93, 6.2%), eGFR (91, 6.0%), and HDL (88, 5.8%).

All four reproductive clustering features have zero NaN remaining — 
confirming the preprocessing pipeline in Steps 1–4 was applied correctly.

**Critically, biomarker missingness is randomly distributed across APO 
subgroups.** The APO distribution among complete cases is virtually 
identical to the full primary sample — never-pregnant women represent 
30.4% vs 29.8%, GDM-only women 7.2% vs 6.9%, and GDM + macrosomia 
1.9% vs 2.0%. The 15.0% reduction in sample size from incomplete cases 
does not introduce meaningful selection bias into the clustering analysis. 
This validates proceeding with complete cases only.

The final effective clustering sample is **n = 1,285 women**, representing 
80.2% of the original analytical sample of 1,603 women. The full exclusion 
cascade is:

- 1,603 — original analytical sample
- − 2 — unknown pregnancy status (Ever_Pregnant NaN)
- − 89 — unknown APO status (APO_Score NaN among ever-pregnant)
- − 227 — incomplete CKM biomarker data
- **= 1,285 — effective clustering sample**

---
## Section 6 · Export

The scaled feature matrix is exported for use in notebook 06. Two files 
are exported:

- **clustering_ready.csv** — the scaled 10-feature matrix for the 1,285 
  complete cases, ready for k-means clustering
- **apo_unknown.csv** — the 89 women with unknown APO status, preserved 
  for descriptive characterization in notebook 07

In [30]:
# ── Prepare final clustering matrix ───────────────────────────────────────
# Complete cases only — 1,285 women, 10 scaled features
df_cluster = df_scaled.dropna().copy()

# Carry forward key descriptive variables from df_pre for notebook 07
carry_forward = [
    'Ever_Pregnant', 'Nulliparous',
    'On_Metformin', 'On_Statin', 'On_Antihypertensive',
    'On_Insulin', 'On_Hormonal_Contraception', 'Any_Prescription',
    'RIDAGEYR', 'RIDRETH3', 'INDFMPIR', 'DMDEDUC2',
    'Waist', 'Mean_DBP', 'WTSAFPRP'
]

for col in carry_forward:
    if col in df_pre.columns:
        df_cluster[col] = df_pre.loc[df_cluster.index, col]

# ── Rebuild df_apo_unknown from original df ────────────────────────────────
# Ever-pregnant women with unknown APO status — preserved for notebook 07
# Must be rebuilt from original df as df_pre had these rows removed
apo_unknown_mask = (df['Ever_Pregnant'] == 1) & (df['APO_Score'].isna())
df_apo_unknown   = df[apo_unknown_mask].copy()

# ── Assertions ─────────────────────────────────────────────────────────────
assert df_cluster.shape[0] == 1285, \
    f'Unexpected row count: {df_cluster.shape[0]}'
assert df_cluster[CLUSTERING_FEATURES].isna().sum().sum() == 0, \
    'NaN present in clustering features'
assert df_cluster.index.name == 'SEQN', \
    'SEQN not index'
assert df_apo_unknown.shape[0] == 89, \
    f'Unexpected APO unknown count: {df_apo_unknown.shape[0]}'

print(f'✓ Clustering matrix validated')
print(f'  Shape: {df_cluster.shape}')
print(f'  Clustering features: {len(CLUSTERING_FEATURES)} (all complete)')
print(f'  Descriptive variables carried forward: '
      f'{len([c for c in carry_forward if c in df_cluster.columns])}')

# ── Export clustering matrix ───────────────────────────────────────────────
OUTPUT_PATH      = Path('../data/processed/clustering_ready.csv')
APO_UNKNOWN_PATH = Path('../data/processed/apo_unknown.csv')

df_cluster.to_csv(OUTPUT_PATH)
print(f'\n✓ clustering_ready.csv exported')
print(f'  Path: {OUTPUT_PATH}')
print(f'  Shape: {df_cluster.shape}')

df_apo_unknown.to_csv(APO_UNKNOWN_PATH)
print(f'\n✓ apo_unknown.csv exported')
print(f'  Path: {APO_UNKNOWN_PATH}')
print(f'  Shape: {df_apo_unknown.shape}')

✓ Clustering matrix validated
  Shape: (1285, 25)
  Clustering features: 10 (all complete)
  Descriptive variables carried forward: 15

✓ clustering_ready.csv exported
  Path: ../data/processed/clustering_ready.csv
  Shape: (1285, 25)

✓ apo_unknown.csv exported
  Path: ../data/processed/apo_unknown.csv
  Shape: (89, 59)


### Export

Two files are exported from this notebook:

**clustering_ready.csv (1,285 × 25):**
The primary output of this notebook — 1,285 complete cases with 10 
RobustScaler-scaled clustering features and 15 descriptive variables 
carried forward for cluster characterization in notebook 07. This file 
is the direct input to notebook 06.

The 10 clustering features are scaled and fully imputed with zero NaN. 
The 15 descriptive variables (APO_Score, Ever_Pregnant, medication flags, 
demographics, Waist, Mean_DBP, and survey weight) are carried forward in 
their original unscaled form for descriptive use only — they are not used 
as clustering inputs.

**apo_unknown.csv (89 × 59):**
The 89 ever-pregnant women excluded from the primary analysis due to 
unknown APO status. Preserved in their original unscaled form for 
descriptive characterization in notebook 07 — examining whether their 
biomarker profiles differ systematically from the primary sample will 
inform whether the exclusion introduced any meaningful bias.

---
## Section 7 · Notebook Summary & Handoff to Notebook 06

This notebook implemented the preprocessing pipeline defined in notebook 
04, transforming the raw CKM feature matrix into a scaled, imputed, 
complete-case matrix ready for clustering.

### Exclusion cascade

| Step | Action | N removed | N remaining |
|---|---|---|---|
| Start | Original analytical sample | — | 1,603 |
| Step 1 | Unknown pregnancy status (Ever_Pregnant NaN) | 2 | 1,601 |
| Step 3 | Unknown APO status (APO_Score NaN) | 89 | 1,512 |
| Section 6 | Incomplete CKM biomarker data | 227 | 1,285 |
| **Final** | **Effective clustering sample** | **—** | **1,285** |

### Preprocessing decisions applied

| Decision | Method | N affected |
|---|---|---|
| Never-pregnant zero-impute | Clinical definition | 451 |
| Parity residual imputation | Median (ever-pregnant) | 15 |
| Pregnancy_Loss residual imputation | Median (ever-pregnant) | 50 |
| Age_Menarche imputation | KNN k=5 | 77 |
| Age_Menarche floor clip | 8.0 years (precocious puberty threshold) | 1 |
| Feature scaling | RobustScaler | All 10 features |

### Final clustering feature set (n=1,285, 10 features)

| Feature | Type | Median | IQR |
|---|---|---|---|
| Age_Menarche | Reproductive | 12.0 | 1.0 |
| Parity | Reproductive | 2.0 | 3.0 |
| Pregnancy_Loss | Reproductive | 0.0 | 1.0 |
| APO_Score | Reproductive | 0.0 | 1.0 |
| HbA1c | CKM biomarker | 5.3% | 0.4 |
| BMI | CKM biomarker | 28.8 kg/m² | 11.4 |
| Mean_SBP | CKM biomarker | 108.0 mmHg | 16.0 |
| eGFR | CKM biomarker | 113.6 mL/min | 20.2 |
| HDL | CKM biomarker | 54.0 mg/dL | 19.0 |
| Glucose | CKM biomarker | 88.0 mg/dL | 11.0 |

### What notebook 06 will do

1. Load `clustering_ready.csv` — 1,285 women, 10 scaled features
2. Run k-means clustering across k=2–8
3. Evaluate solutions using elbow method and silhouette scores
4. Run hierarchical clustering as sensitivity analysis
5. Select optimal k based on metrics and clinical interpretability
6. Run sensitivity analyses:
   - StandardScaler vs RobustScaler — confirm scaling choice is not 
     driving cluster structure
   - APO_Score=0 assumption for excluded 89 women — confirm primary 
     cluster structure is robust
7. Export cluster assignments for characterization in notebook 07